In [ ]:
from pathlib import Path
 
DATA_PROC_DIR = Path(r"C:\Users\jorge gonzalez\Documents\TFG\Proyecto\data\processed")
REPORTS_DIR   = Path(r"C:\Users\jorge gonzalez\Documents\TFG\Proyecto\reports")
FIGS_DIR      = REPORTS_DIR / "figuras"
 
for d in [DATA_PROC_DIR, REPORTS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
 
print("DATA_PROC_DIR:", DATA_PROC_DIR)

DATA_PROC_DIR: C:\Users\jorge gonzalez\Documents\TFG\Proyecto_clean\data\processed


Cargar parquet

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet(DATA_PROC_DIR / "livvo_clean_all_v2.parquet")

df.shape

(10108504, 9)

Conversión de fechas

In [3]:
df["date"] = pd.to_datetime(df["date"])
df["fechaimport"] = pd.to_datetime(df["fechaimport"])

Eliminar registros con fechaimport < date

In [4]:
invalid = df["fechaimport"] < df["date"]

print("Filas con fechaimport < date:", invalid.sum())
print("Porcentaje:", round(invalid.mean()*100,2), "%")

df = df.loc[~invalid].copy()

df.shape

Filas con fechaimport < date: 5161089
Porcentaje: 51.06 %


(4947415, 9)

In [5]:
df["diff_days"] = (df["fechaimport"] - df["date"]).dt.days
df["diff_days"].describe()

count    4.947415e+06
mean     1.211707e+02
std      8.546947e+01
min      0.000000e+00
25%      4.900000e+01
50%      1.070000e+02
75%      1.820000e+02
max      3.650000e+02
Name: diff_days, dtype: float64

In [6]:
df = df.drop(columns=["diff_days"])

He analizado la variable fechaimport para comprobar la coherencia temporal de los datos. Detecté que una parte importante de los registros presentaba fechaimport < date, lo que implica que el sistema registraba la información antes de la fecha real de la estancia.

Para evitar inconsistencias temporales en el dataset, decidí eliminar estos registros durante la fase de limpieza de datos. De este modo, el conjunto final mantiene únicamente observaciones con una relación temporal coherente entre ambas variables.

Aunque fechaimport se utiliza en esta fase para asegurar la calidad del dataset y seleccionar la versión más reciente de cada registro, la variable principal utilizada en el análisis y en los modelos de predicción es date, que representa la fecha real de la estancia.

Eliminar agregados GLOBAL

In [7]:
keys = ["codigo_hotel","codigo_ttoo","date"]

duplicates = df.duplicated(keys).sum()

print("Duplicados encontrados:", duplicates)

df = df.drop_duplicates(keys, keep="last")

df.shape

Duplicados encontrados: 4920842


(26573, 9)

In [8]:
keys = ["date","codigo_hotel","codigo_ttoo"]

grp_sum = df.groupby(keys)[["roomnights","bednights"]].sum().rename(
    columns={"roomnights":"rn_sum","bednights":"bn_sum"}
)

tmp = df.merge(grp_sum, on=keys, how="left")
sizes = df.groupby(keys).size().rename("n").reset_index()
tmp = tmp.merge(sizes, on=keys, how="left")

cond = (tmp["n"]>1) & (
    np.isclose(tmp["roomnights"], tmp["rn_sum"], equal_nan=False) |
    np.isclose(tmp["bednights"], tmp["bn_sum"], equal_nan=False)
)

df_global = df.loc[~cond.values].copy()
df_global.shape

(26573, 9)

Filtrar por última fechaimport (Transformación 1)

In [9]:
df_global["year"] = df_global["date"].dt.year

df_t1 = (
    df_global
    .sort_values("fechaimport")
    .groupby(["codigo_hotel","codigo_ttoo","date","year"], as_index=False)
    .tail(1)
)

df_t1.shape

(26573, 10)

Reglas estrictas de stock (Transformación 2)

In [10]:
STOCK_REAL = {
    "Hotel_1":  94,
    "Hotel_2": 138,
    "Hotel_3": 251,
}

def apply_stock_rules_strict(df, stock_real):
    df = df.copy()
    df["codigo_hotel"] = df["codigo_hotel"].str.upper()
    expected = df["codigo_hotel"].map(stock_real)

    # UNDER (stock incorrecto) → eliminar
    under = df["stock"] < expected
    df = df.loc[~under].copy()

    # Recalcular expected
    expected = df["codigo_hotel"].map(stock_real)

    # OVER → capar
    over = df["stock"] > expected
    df.loc[over, "stock"] = expected.loc[over]

    # STOCK nulo/0 → sustituir por stock real
    missing = df["stock"].isna() | (df["stock"] == 0)
    df.loc[missing, "stock"] = expected.loc[missing]

    return df

df_t2 = apply_stock_rules_strict(df_t1, STOCK_REAL)
df_t2.shape

(26573, 10)

Agregado día–hotel–ttoo

In [11]:
agg_ttoo = (
    df_t2.groupby(["date","codigo_hotel","codigo_ttoo"], dropna=False)
         .agg({
             "roomnights":"sum",
             "bednights":"sum",
             "neto":"sum",
             "stock":"max"
         })
         .reset_index()
)

agg_ttoo.shape

(26573, 7)

Agregado día–hotel (ocupación y ADR)

In [12]:
agg_hotel = (
    agg_ttoo.groupby(["date","codigo_hotel"], dropna=False)
            .agg({
                "roomnights":"sum",
                "bednights":"sum",
                "neto":"sum",
                "stock":"max"
            })
            .reset_index()
)

agg_hotel["ocup_total"] = agg_hotel["roomnights"] / agg_hotel["stock"]
agg_hotel["ADR"] = agg_hotel["neto"] / agg_hotel["roomnights"]

agg_hotel.head()

,date,codigo_hotel,roomnights,bednights,neto,stock,ocup_total,ADR
0,2023-01-01,HOTEL_1,48,97,5736.42,94,0.510638,119.508750
1,2023-01-01,HOTEL_2,112,227,25226.37,138,0.811594,225.235446
2,2023-01-01,HOTEL_3,154,328,41940.92,251,0.613546,272.343636
3,2023-01-02,HOTEL_1,62,116,6313.73,94,0.659574,101.834355
4,2023-01-02,HOTEL_2,114,232,25242.56,138,0.826087,221.425965


Guardar resultados finales

In [13]:
agg_ttoo.to_parquet(DATA_PROC_DIR / "livvo_day_ttoo_final.parquet", index=False)
agg_hotel.to_parquet(DATA_PROC_DIR / "livvo_day_hotel_final.parquet", index=False)

"OK"

'OK'